# Angle-resolved EL scan analysis

Loads scans saved by the goniometer GUI (`ScanWorker._save`) and plots everything they contain,
in Nature figure style (89 mm / 183 mm widths, 5–7 pt sans-serif, editable-text PDF output).

Each scan is two files:

* `<name>_<timestamp>.txt`: wavelength row, then the **raw** block, then the **dark-corrected** block (one row per angle)
* `<name>_sourcemeter_<timestamp>.txt`: angle, measured voltage, measured current, and the compliance flag

The source-meter file is found automatically. Set the options in the next cell and run all cells.

In [ ]:
from pathlib import Path

# ── Settings ────────────────────────────────────────────────────────
DATA_DIR      = Path("data")       # where the GUI saves scans (config.yaml → data.default_save_dir)
DATA_FILES    = None               # None = every spectrum file in DATA_DIR, or a list of paths
SPECTRUM      = "corr"             # "corr" (dark-corrected) or "raw"
NORMAL_ANGLE  = 0.0                # goniometer angle (deg) of the surface normal, for the Lambertian reference
SMOOTH_PX     = 7                  # moving-average window (pixels) used only for peak / FWHM extraction
MIN_SNR       = 10                 # angles whose peak is below MIN_SNR × noise get no peak / centroid / FWHM / normalized spectrum
SIGNAL_FRAC   = 0.05               # wavelengths where the mean spectrum < this fraction of its max are ignored for the centroid
WL_RANGE      = None               # e.g. (450, 750) to crop the wavelength axis of the spectral plots
SAVE_DIR      = Path("figures")
SAVE_FORMATS  = ("pdf", "png")     # () to skip saving

In [ ]:
import logging, re
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from cycler import cycler

%matplotlib inline
%config InlineBackend.figure_format = "retina"
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ── Nature style ────────────────────────────────────────────────────
MM = 1 / 25.4
SINGLE, ONE_HALF, DOUBLE = 89 * MM, 120 * MM, 183 * MM    # Nature column widths
MAX_H = 170 * MM

# Okabe–Ito colour-blind-safe palette
OI = dict(blue="#0072B2", orange="#E69F00", green="#009E73", vermillion="#D55E00",
          purple="#CC79A7", sky="#56B4E9", yellow="#F0E442", black="#000000")
ANGLE_CMAP = "viridis"     # colour = angle
MAP_CMAP   = "magma"       # angle–wavelength intensity map
COMP_KW    = dict(marker="x", color=OI["vermillion"], s=14, lw=0.7, zorder=5, label="In compliance")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "mathtext.default": "regular",
    "axes.linewidth": 0.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelpad": 2, "axes.prop_cycle": cycler(color=[OI[k] for k in
                        ("blue", "vermillion", "green", "orange", "purple", "sky", "black")]),
    "lines.linewidth": 0.8, "lines.markersize": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.5, "ytick.minor.size": 1.5,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.minor.width": 0.4, "ytick.minor.width": 0.4,
    "xtick.major.pad": 2, "ytick.major.pad": 2,
    "axes.formatter.use_mathtext": True, "axes.formatter.limits": (-3, 4),
    "legend.frameon": False, "legend.handlelength": 1.4, "legend.borderaxespad": 0.3,
    "image.cmap": ANGLE_CMAP,
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",   # editable text
})

def panel_labels(axes, start="a", dx=-14, dy=4):
    # bold lower-case panel letters at the top-left outside each axis (Nature convention)
    for k, ax in enumerate(np.ravel(axes)):
        ax.annotate(chr(ord(start) + k), xy=(0, 1), xycoords="axes fraction",
                    xytext=(dx, dy), textcoords="offset points",
                    fontsize=8, fontweight="bold", ha="left", va="bottom")

def save(fig, name):
    if not SAVE_FORMATS:
        return
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    for ext in SAVE_FORMATS:
        fig.savefig(SAVE_DIR / f"{name}.{ext}")

def angle_colorbar(fig, ax, ang, **kw):
    sm = mpl.cm.ScalarMappable(cmap=ANGLE_CMAP, norm=Normalize(ang.min(), ang.max()))
    cb = fig.colorbar(sm, ax=ax, **{"pad": 0.02, "aspect": 30, **kw})
    cb.set_label("Angle (°)")
    cb.outline.set_linewidth(0.5); cb.ax.tick_params(width=0.5, length=2)
    return cb

## Loading

Same file format as the GUI's Plot tab (`gui/tabs/plot_tab.py`), re-implemented here without the PyQt dependency.
Rows are sorted by angle. Spectral quantities are derived from `SPECTRUM`:

| key | meaning |
|---|---|
| `integrated` | ∫ I(λ) dλ (trapezoid over the full wavelength range) |
| `peak_wl`, `peak` | wavelength and height of the maximum of the smoothed spectrum |
| `centroid` | intensity-weighted mean wavelength over the emission band |
| `fwhm` | full width at half maximum of the smoothed spectrum (interpolated) |
| `per_current` | `integrated / I`: emission per unit current, which removes current drift |
| `snr`, `valid` | peak / off-band noise, and whether it is ≥ `MIN_SNR`. Peak, centroid, FWHM and normalized spectra are only shown where valid |
| `lambert` | cos(θ − θ_normal) scaled to the integrated intensity at the angle closest to normal |

In [ ]:
def _delim(line):
    return next((d for d in (";", "\t", ",") if d in line), "\t")

def _read(path):
    meta, data = {}, []
    for line in Path(path).read_text().splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            if ":" in s:
                k, v = s[1:].split(":", 1)
                meta[k.strip()] = v.strip()
            continue
        data.append(s)
    return meta, data

def sourcemeter_path(path):
    path = Path(path)
    m = re.match(r"^(.+?)_(\d{8}_\d{6})$", path.stem)
    name = f"{m.group(1)}_sourcemeter_{m.group(2)}" if m else f"{path.stem}_sourcemeter"
    return path.with_name(name + path.suffix)

def load_scan(path):
    path = Path(path)
    meta, lines = _read(path)
    if len(lines) < 3:
        raise ValueError(f"{path.name}: no data")
    d = _delim(lines[0])
    wl = np.array(lines[0].split(d)[1:], float)
    body = np.array([l.split(d)[:1 + wl.size] for l in lines[1:]], float)
    if len(body) % 2:
        raise ValueError(f"{path.name}: expected raw + corrected blocks, got {len(body)} rows")
    n = len(body) // 2
    ang, raw, corr = body[:n, 0], body[:n, 1:], body[n:, 1:]

    v = np.full(n, np.nan); i = np.full(n, np.nan); comp = np.zeros(n, bool); sm_meta = {}
    sp = sourcemeter_path(path)
    if sp.is_file():
        sm_meta, sm_lines = _read(sp)
        rows = []
        for l in sm_lines:
            parts = l.split(_delim(l))
            try:
                rows.append([float(x) for x in parts[:4]])
            except ValueError:
                continue      # column-name row
        sm = np.array(rows[:n])
        v[:len(sm)], i[:len(sm)], comp[:len(sm)] = sm[:, 1], sm[:, 2], sm[:, 3].astype(bool)

    o = np.argsort(ang, kind="stable")
    return dict(name=path.stem, path=path, meta=meta, sm_meta=sm_meta, has_sm=sp.is_file(),
                wl=wl, ang=ang[o], raw=raw[o], corr=corr[o], v=v[o], i=i[o], comp=comp[o])

def _smooth(y, w):
    if w <= 1:
        return y
    k = np.ones(w) / w
    return np.apply_along_axis(lambda r: np.convolve(r, k, mode="same"), -1, y)

def _fwhm(wl, y):
    j = int(np.argmax(y)); half = y[j] / 2
    if y[j] <= 0:
        return np.nan
    l = j
    while l > 0 and y[l] > half: l -= 1
    r = j
    while r < len(y) - 1 and y[r] > half: r += 1
    if y[l] > half or y[r] > half:          # band runs off the detector edge
        return np.nan
    xl = np.interp(half, [y[l], y[l + 1]], [wl[l], wl[l + 1]])
    xr = np.interp(half, [y[r], y[r - 1]], [wl[r], wl[r - 1]])
    return xr - xl

def analyse(s, which=SPECTRUM):
    sp, wl, ang = s[which], s["wl"], s["ang"]
    trapz = getattr(np, "trapezoid", None) or np.trapz
    s["sp"], s["sp_label"] = sp, ("Raw" if which == "raw" else "Dark-corrected")
    s["integrated"] = trapz(sp, wl, axis=1)
    sm = _smooth(sp, SMOOTH_PX)
    k = sm.argmax(axis=1)
    s["peak_wl"], s["peak"] = wl[k], sm[np.arange(len(ang)), k]
    band = sp.mean(0) >= SIGNAL_FRAC * sp.mean(0).max()
    w = np.clip(sp[:, band], 0, None)
    s["centroid"] = (w * wl[band]).sum(1) / np.where(w.sum(1) > 0, w.sum(1), np.nan)
    s["fwhm"] = np.array([_fwhm(wl, r) for r in sm])
    off = ~band if (~band).sum() > 10 else np.ones_like(band)
    s["noise"] = np.std(sp[:, off] - _smooth(sp[:, off], 15), axis=1)        # high-frequency noise off the band
    s["snr"] = s["peak"] / np.where(s["noise"] > 0, s["noise"], np.nan)
    s["valid"] = s["snr"] >= MIN_SNR
    for key in ("peak_wl", "centroid", "fwhm"):
        s[key] = np.where(s["valid"], s[key], np.nan)
    s["per_current"] = s["integrated"] / s["i"]
    j0 = int(np.argmin(np.abs(ang - NORMAL_ANGLE)))
    s["j_normal"] = j0
    s["lambert"] = s["integrated"][j0] * np.clip(np.cos(np.deg2rad(ang - NORMAL_ANGLE)), 0, None)
    s["dark"] = s["raw"] - s["corr"]
    return s

files = [Path(p) for p in DATA_FILES] if DATA_FILES else sorted(DATA_DIR.glob("*.txt"))
files = [p for p in files if "_sourcemeter" not in p.stem]
scans = []
for p in files:
    try:
        scans.append(analyse(load_scan(p)))
    except Exception as e:
        print(f"skipped {p.name}: {e}")
print(f"{len(scans)} scan(s) loaded from {DATA_DIR if not DATA_FILES else 'DATA_FILES'}")

## Summary

In [ ]:
for k, s in enumerate(scans):
    r, c = np.unravel_index(np.argmax(s["sp"]), s["sp"].shape)
    print(f"[{k}] {s['name']}")
    print(f"    angles      {len(s['ang'])}  ({s['ang'].min():g} to {s['ang'].max():g}°)")
    print(f"    wavelengths {len(s['wl'])}  ({s['wl'][0]:.1f}–{s['wl'][-1]:.1f} nm)")
    print(f"    max         {s['sp'][r, c]:.0f} counts at {s['wl'][c]:.1f} nm, {s['ang'][r]:g}°")
    print(f"    brightest   {s['ang'][np.argmax(s['integrated'])]:g}° (integrated)")
    print(f"    peak λ      {np.nanmin(s['peak_wl']):.1f}–{np.nanmax(s['peak_wl']):.1f} nm")
    if s["has_sm"]:
        print(f"    V           {np.nanmin(s['v']):.3f}–{np.nanmax(s['v']):.3f} V")
        print(f"    I           {np.nanmin(s['i'])*1e3:.3f}–{np.nanmax(s['i'])*1e3:.3f} mA")
        print(f"    compliance  {s['comp'].sum()}/{len(s['comp'])} points")
    else:
        print("    no source-meter file found")
    for key, val in {**s["meta"], **s["sm_meta"]}.items():
        print(f"      {key:<18} {val}")

## Figures for one scan

Pick the scan with `SCAN`, which is an index into `scans`. Each figure is saved as `<scan name>_<figure>.{pdf,png}`.
Points measured in source-meter compliance are marked × in every angle plot.

In [ ]:
SCAN = 0
s = scans[SCAN]
wl_mask = np.ones_like(s["wl"], bool) if WL_RANGE is None else (s["wl"] >= WL_RANGE[0]) & (s["wl"] <= WL_RANGE[1])

def mark_compliance(ax, x, y, s):
    if s["comp"].any():
        ax.scatter(x[s["comp"]], y[s["comp"]], **COMP_KW)

### Figure 1: spectra
**a** Spectrum at every angle, coloured by angle. **b** The same spectra each normalised to its maximum, which shows changes in shape (only angles with SNR ≥ `MIN_SNR`). **c** Angle–wavelength intensity map.

In [ ]:
def fig_spectra(s):
    wl, ang, sp = s["wl"][wl_mask], s["ang"], s["sp"][:, wl_mask]
    cols = mpl.colormaps[ANGLE_CMAP](Normalize(ang.min(), ang.max())(ang))
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 52 * MM), layout="constrained")

    for y, c in zip(sp, cols):
        ax[0].plot(wl, y, color=c, lw=0.6)
    ax[0].set(xlabel="Wavelength (nm)", ylabel=f"{s['sp_label']} intensity (counts)")
    ax[0].set_xlim(wl[0], wl[-1])

    peak = sp.max(1, keepdims=True)
    norm = np.divide(sp, peak, out=np.zeros_like(sp), where=peak > 0)
    for y, c, ok in zip(norm, cols, s["valid"]):
        if ok:
            ax[1].plot(wl, y, color=c, lw=0.6)
    ax[1].set(xlabel="Wavelength (nm)", ylabel="Normalized intensity", ylim=(-0.05, 1.08))
    ax[1].set_xlim(wl[0], wl[-1])
    cb = angle_colorbar(fig, ax[:2], ang, location="top", shrink=0.5, aspect=40)
    cb.ax.xaxis.set_label_position("top")

    im = ax[2].pcolormesh(wl, ang, sp, shading="nearest", cmap=MAP_CMAP, rasterized=True)
    ax[2].set(xlabel="Wavelength (nm)", ylabel="Angle (°)")
    cb = fig.colorbar(im, ax=ax[2], pad=0.02, aspect=30)
    cb.set_label("Intensity (counts)"); cb.outline.set_linewidth(0.5); cb.ax.tick_params(width=0.5, length=2)
    for sp_ in ax[2].spines.values(): sp_.set_visible(True)

    panel_labels(ax)
    return fig

fig = fig_spectra(s); save(fig, f"{s['name']}_spectra"); plt.show()

### Figure 2: angular emission
**a** Integrated intensity against angle, with the Lambertian (cos θ) reference. **b** Peak wavelength and centroid. **c** FWHM. **d** Integrated intensity on a polar plot, normalised to the value at the normal, with the Lambertian reference dashed.

In [ ]:
def fig_angular(s):
    ang, I, L = s["ang"], s["integrated"], s["lambert"]
    I0 = I[s["j_normal"]]
    fig = plt.figure(figsize=(DOUBLE, 110 * MM), layout="constrained")
    gs = fig.add_gridspec(3, 2, width_ratios=[1, 1.25])
    ax_i = fig.add_subplot(gs[0, 0])
    ax_l = fig.add_subplot(gs[1, 0], sharex=ax_i)
    ax_w = fig.add_subplot(gs[2, 0], sharex=ax_i)
    ax_p = fig.add_subplot(gs[:, 1], projection="polar")

    ax_i.plot(ang, I, "o-", color=OI["blue"], ms=2.5, label="Measured")
    ax_i.plot(ang, L, "--", color=OI["black"], lw=0.6, label="Lambertian")
    mark_compliance(ax_i, ang, I, s)
    ax_i.set_ylabel("Integrated intensity\n(counts nm)")
    ax_i.legend(loc="lower center")

    ax_l.plot(ang, s["peak_wl"], "o-", color=OI["vermillion"], ms=2.5, label="Peak")
    ax_l.plot(ang, s["centroid"], "s-", color=OI["green"], ms=2, label="Centroid")
    ax_l.set_ylabel("Wavelength (nm)")
    ax_l.legend(loc="best", ncols=2)

    ax_w.plot(ang, s["fwhm"], "^-", color=OI["purple"], ms=2.5)
    ax_w.set(xlabel="Angle (°)", ylabel="FWHM (nm)")
    for a in (ax_i, ax_l):
        a.tick_params(labelbottom=False)
    fig.align_ylabels([ax_i, ax_l, ax_w])

    th = np.deg2rad(ang - NORMAL_ANGLE)
    ax_p.plot(th, I / I0, "o-", color=OI["blue"], ms=2.5)
    tt = np.linspace(th.min(), th.max(), 361)
    ax_p.plot(tt, np.clip(np.cos(tt), 0, None), "--", color=OI["black"], lw=0.6)
    if s["comp"].any():
        ax_p.scatter(th[s["comp"]], (I / I0)[s["comp"]], **COMP_KW)
    ax_p.set_theta_zero_location("N"); ax_p.set_theta_direction(-1)
    ax_p.set_thetamin(np.rad2deg(th.min())); ax_p.set_thetamax(np.rad2deg(th.max()))
    ax_p.set_rlim(0, max(1.05, np.nanmax(I / I0) * 1.05))
    ax_p.grid(lw=0.3, alpha=0.6); ax_p.spines["polar"].set_linewidth(0.5)
    ax_p.tick_params(labelsize=6, pad=1)
    ax_p.set_xlabel("Normalized integrated intensity", labelpad=-40)

    panel_labels([ax_i, ax_l, ax_w], dx=-30); panel_labels([ax_p], start="d", dx=-4)
    return fig

fig = fig_angular(s); save(fig, f"{s['name']}_angular"); plt.show()

### Figure 3: electrical
**a** Measured voltage. **b** Measured current. **c** Integrated emission per unit current, which separates real angular dependence from current drift during the scan.

In [ ]:
def fig_electrical(s):
    if not s["has_sm"]:
        print("No source-meter file for this scan; skipping.")
        return None
    ang = s["ang"]
    fig, ax = plt.subplots(3, 1, figsize=(SINGLE, 95 * MM), sharex=True, layout="constrained")
    ax[0].plot(ang, s["v"], "o-", color=OI["green"], ms=2.5); mark_compliance(ax[0], ang, s["v"], s)
    ax[0].set_ylabel("Voltage (V)")
    ax[1].plot(ang, s["i"] * 1e3, "o-", color=OI["orange"], ms=2.5); mark_compliance(ax[1], ang, s["i"] * 1e3, s)
    ax[1].set_ylabel("Current (mA)")
    ax[2].plot(ang, s["per_current"], "o-", color=OI["blue"], ms=2.5); mark_compliance(ax[2], ang, s["per_current"], s)
    ax[2].set(xlabel="Angle (°)", ylabel="Intensity / current\n(counts nm A$^{-1}$)")
    if s["comp"].any():
        ax[0].legend(loc="best")
    fig.align_ylabels(ax)
    panel_labels(ax, dx=-24)
    return fig

fig = fig_electrical(s)
if fig: save(fig, f"{s['name']}_electrical"); plt.show()

### Figure 4: raw data and dark subtraction
**a** Raw against dark-corrected spectrum at the brightest angle. **b** The subtracted background (raw − corrected): its mean over all angles, with the min–max band showing how much it drifts. It reads low where the corrected spectrum was clipped at zero. **c** Raw angle–wavelength map.

In [ ]:
def fig_raw(s):
    wl, ang = s["wl"][wl_mask], s["ang"]
    raw, corr, dark = s["raw"][:, wl_mask], s["corr"][:, wl_mask], s["dark"][:, wl_mask]
    j = int(np.argmax(s["integrated"]))
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 52 * MM), layout="constrained")

    ax[0].plot(wl, raw[j], color=OI["black"], lw=0.6, label="Raw")
    ax[0].plot(wl, corr[j], color=OI["blue"], lw=0.6, label="Dark-corrected")
    ax[0].set(xlabel="Wavelength (nm)", ylabel="Intensity (counts)", xlim=(wl[0], wl[-1]))
    ax[0].legend(title=f"{ang[j]:g}°", loc="upper right", title_fontsize=6)

    ax[1].fill_between(wl, dark.min(0), dark.max(0), color=OI["sky"], alpha=0.35, lw=0, label="Min–max over angles")
    ax[1].plot(wl, dark.mean(0), color=OI["blue"], lw=0.6, label="Mean")
    ax[1].set(xlabel="Wavelength (nm)", ylabel="Raw − corrected (counts)", xlim=(wl[0], wl[-1]))
    ax[1].legend(loc="best")

    im = ax[2].pcolormesh(wl, ang, raw, shading="nearest", cmap=MAP_CMAP, rasterized=True)
    ax[2].set(xlabel="Wavelength (nm)", ylabel="Angle (°)")
    cb = fig.colorbar(im, ax=ax[2], pad=0.02, aspect=30)
    cb.set_label("Raw intensity (counts)"); cb.outline.set_linewidth(0.5); cb.ax.tick_params(width=0.5, length=2)
    for sp_ in ax[2].spines.values(): sp_.set_visible(True)

    panel_labels(ax)
    return fig

fig = fig_raw(s); save(fig, f"{s['name']}_raw"); plt.show()

### Batch: every figure for every scan
Run this cell to export all four figures for every loaded scan without displaying them.

In [ ]:
for sc in scans:
    for tag, fn in [("spectra", fig_spectra), ("angular", fig_angular),
                    ("electrical", fig_electrical), ("raw", fig_raw)]:
        f = fn(sc)
        if f is not None:
            save(f, f"{sc['name']}_{tag}"); plt.close(f)
print(f"saved to {SAVE_DIR.resolve()}" if SAVE_FORMATS else "SAVE_FORMATS is empty, nothing saved")

## Comparing scans
Shown only when more than one scan is loaded. **a** Integrated intensity normalised at the normal. **b** Peak wavelength. **c** Spectrum closest to the normal, normalised. **d** Emission per unit current, normalised at the normal.

In [ ]:
def fig_compare(scans):
    fig, ax = plt.subplots(2, 2, figsize=(DOUBLE, 100 * MM), layout="constrained")
    ax = ax.ravel()
    for k, sc in enumerate(scans):
        c = mpl.rcParams["axes.prop_cycle"].by_key()["color"][k % 7]
        j0, ang = sc["j_normal"], sc["ang"]
        ax[0].plot(ang, sc["integrated"] / sc["integrated"][j0], "o-", ms=2, color=c, label=sc["name"])
        ax[1].plot(ang, sc["peak_wl"], "o-", ms=2, color=c)
        y = sc["sp"][j0]; ax[2].plot(sc["wl"], y / y.max(), color=c, lw=0.7)
        if sc["has_sm"]:
            pc = sc["per_current"]; ax[3].plot(ang, pc / pc[j0], "o-", ms=2, color=c)
    th = np.linspace(-90, 90, 361)
    ax[0].plot(th + NORMAL_ANGLE, np.cos(np.deg2rad(th)), "--", color=OI["black"], lw=0.6, label="Lambertian")
    ax[0].set(xlabel="Angle (°)", ylabel="Normalized integrated intensity")
    ax[1].set(xlabel="Angle (°)", ylabel="Peak wavelength (nm)")
    ax[2].set(xlabel="Wavelength (nm)", ylabel="Normalized intensity")
    if WL_RANGE: ax[2].set_xlim(*WL_RANGE)
    ax[3].set(xlabel="Angle (°)", ylabel="Normalized intensity / current")
    ax[0].legend(loc="best", fontsize=5)
    panel_labels(ax)
    return fig

if len(scans) > 1:
    fig = fig_compare(scans); save(fig, "comparison"); plt.show()
else:
    print("Only one scan is loaded; nothing to compare.")

## Export derived quantities
One CSV per scan with every per-angle quantity used above, for plotting in other software.

In [ ]:
for sc in scans:
    cols = ["ang", "v", "i", "comp", "integrated", "per_current", "lambert", "peak_wl", "peak", "centroid", "fwhm", "snr"]
    hdr = "angle_deg,voltage_V,current_A,in_compliance,integrated_counts_nm,integrated_per_A,lambertian_ref," \
          "peak_wl_nm,peak_counts,centroid_nm,fwhm_nm,snr"
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    np.savetxt(SAVE_DIR / f"{sc['name']}_derived.csv", np.column_stack([sc[c].astype(float) for c in cols]),
               delimiter=",", header=hdr, comments="", fmt="%.6g")
print("done")